# 7. 멀티 에이전트 고도화 — 하네스 엔지니어링

| 구분 | 내용 |
|------|------|
| **목표** | Deep Agents 하네스 아키텍처를 이해하고, SKILL.md를 활용하여 에이전트 행동을 선언적으로 제어한다 |
| **키워드** | AgentHarness, SKILL.md, AGENTS.md, Progressive Disclosure, BackendProtocol, CompositeBackend, Sandbox, ACP |

---

### 이 노트북에서 배우는 것

| Part | 주제 | 핵심 내용 |
|------|------|-----------|
| **Part 1** | SKILL.md 설계 + 하네스 아키텍처 | 계획(write_todos) + 도구(가상 파일시스템 7종) + 메모리 + 컨텍스트 오프로딩 |
| **Part 2** | 스토리지 백엔드 + 샌드박스 + 배포 | State/Filesystem/Store/Composite 백엔드, Sandbox-as-Tool, ACP |

## 0) 환경 설정

In [1]:
# [7-1] : 라이브러리 설치
# [주의] 이미 설치되어 있다면 이 셀을 건너뛰세요
!uv pip install -q deepagents langchain langchain-openai langgraph python-dotenv

In [1]:
# [7-2] : 환경변수 로드 및 API 키 검증
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("API 키 검증 완료")

API 키 검증 완료


In [2]:
# [7-4] : 공통 LLM 모델 초기화
# [핵심] 이 코드는 하네스 엔지니어링의 핵심 구성 요소를 구현
# [주의] 실행 전 API Key 설정을 확인하세요
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")
print(f"모델 설정 완료: {model.model_name}")

모델 설정 완료: gpt-5.4-mini


---
# Part 1 : SKILL.md 설계 + 하네스 아키텍처

## [1-1] : Agent 하네스란?

**AgentHarness**는 장기 실행 자율 에이전트를 위한 **포괄적 기능 제공자**입니다.  
에이전트에게 필요한 모든 인프라(계획, 도구, 메모리, 컨텍스트)를 하나로 조립하여 제공합니다.

### 하네스 = 에이전트의 운영체제

| 하네스 구성 요소 | 역할 | 비유 |
|-----------------|------|------|
| **Planning** | 구조화된 태스크 리스트 관리 | OS의 프로세스 스케줄러 |
| **Filesystem** | 가상/로컬 파일 읽기, 쓰기, 검색 | OS의 파일시스템 |
| **Memory** | 영속적 컨텍스트 (AGENTS.md) | OS의 환경변수 |
| **Skills** | 온디맨드 전문 지식 (SKILL.md) | OS의 앱 스토어 |
| **Context Management** | 오프로딩, 요약을 통한 컨텍스트 압축 | OS의 메모리 관리 |
| **Code Execution** | 샌드박스 환경에서 안전한 코드 실행 | OS의 격리 프로세스 |
| **Human-in-the-Loop** | 민감 작업에 대한 사람 승인 | OS의 권한 요청 |

`create_deep_agent()`를 호출하면 이 모든 기능이 **자동으로 조립**되어 하나의 에이전트로 제공됩니다.

```
┌─────────────────────────────────────────┐
│            AgentHarness                 │
│  ┌──────────┐  ┌──────────┐            │
│  │ Planning │  │ Memory   │            │
│  │write_todos│  │AGENTS.md │            │
│  └──────────┘  └──────────┘            │
│  ┌──────────┐  ┌──────────┐            │
│  │Filesystem│  │ Skills   │            │
│  │ 7종 도구  │  │ SKILL.md │            │
│  └──────────┘  └──────────┘            │
│  ┌──────────┐  ┌──────────┐            │
│  │ Context  │  │ Sandbox  │            │
│  │Offloading│  │ Execute  │            │
│  └──────────┘  └──────────┘            │
└─────────────────────────────────────────┘
```

## [1-2] : 하네스 구성 요소 상세

### 1) Planning — `write_todos`

에이전트는 `write_todos` 도구를 사용하여 복잡한 작업을 **구조화된 태스크 리스트**로 분해합니다.  
각 태스크는 `pending` → `in_progress` → `completed` 상태를 가집니다.

### 2) Filesystem — 7종 도구

| 도구 | 설명 | 예시 |
|------|------|------|
| `ls` | 디렉토리 목록 (메타데이터 포함) | `ls(path="/project/src")` |
| `read_file` | 파일 내용 읽기 (줄 번호 포함) | `read_file(path="/main.py")` |
| `write_file` | 새 파일 생성 | `write_file(path="/config.yaml", content="...")` |
| `edit_file` | 문자열 치환 편집 | `edit_file(path="/main.py", old="v1", new="v2")` |
| `glob` | 패턴 기반 파일 검색 | `glob(pattern="**/*.py")` |
| `grep` | 내용 검색 (여러 출력 모드) | `grep(pattern="TODO", path="/src")` |
| `execute` | 쉘 명령 실행 (샌드박스 전용) | `execute(command="pytest tests/")` |

### 3) Context Management — 오프로딩과 요약

| 기법 | 동작 | 트리거 |
|------|------|--------|
| **오프로딩** | 20,000 토큰 초과 콘텐츠를 디스크에 저장, 포인터 참조 유지 | 콘텐츠 크기 기준 |
| **요약** | 대화 히스토리를 구조화된 요약으로 압축 | 모델 윈도우 한계 접근 시 |

> **[핵심]** 원본 메시지는 파일시스템 스토리지에 보존되므로 정보 손실이 없습니다.

In [3]:
from pathlib import Path
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# 파일이 저장될 로컬 디렉터리
workspace = Path("agent_workspace").resolve()
workspace.mkdir(parents=True, exist_ok=True)
workspace

WindowsPath('C:/Users/User/hynix_agent/agent_workspace')

In [17]:
from langchain_core.tools import tool

# 딥 에이전트 생성 - 파일 다루기
@tool
def analyze_data(data_description: str) -> str:
    """데이터를 분석하고 통계 요약을 반환합니다."""
    return f"'{data_description}'"

agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(
        root_dir=workspace,
        virtual_mode=True
    ),
    subagents=[
        {
            "name": "data-analyzer",
            "description": "수집된 데이터를 분석하여 핵심 인사이트를 추출합니다.",
            "system_prompt": (
                "당신은 데이터 분석 전문가입니다.\n"
                "제공된 데이터에서 패턴, 트렌드, 핵심 인사이트를 추출합니다.\n"
                "분석 결과를 불릿 포인트로 정리하세요."
            ),
            "tools": [analyze_data],
        },
        {
            "name": "report-writer",
            "description": "분석 결과를 바탕으로 전문적인 보고서를 작성합니다.",
            "system_prompt": (
                "당신은 테크니컬 라이터입니다.\n"
                "분석 결과를 바탕으로 명확하고 읽기 쉬운 보고서를 작성합니다.\n"
                "보고서 구조: 개요 -> 핵심 발견 -> 결론"
            ),
            "tools": [],
        }],
    system_prompt="사용자 요청에 따라 파일을 다루는 도구를 사용해라."
)

In [18]:
from langchain.messages import SystemMessage, AIMessage, HumanMessage
result = agent.invoke({"messages":HumanMessage(content="다음 내용을 hello.txt 파일로 만들어줘. 내용:'Deep Agent가 만든 파일입니다.'")})
result

{'messages': [HumanMessage(content="다음 내용을 hello.txt 파일로 만들어줘. 내용:'Deep Agent가 만든 파일입니다.'", additional_kwargs={}, response_metadata={}, id='5b76816d-75dc-4d41-86b2-c6969eac1362'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 5505, 'total_tokens': 5535, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Ds0PMiVO8INinR53kc98Llm2hoMKY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed963-baad-76a2-851e-75e62ce5e150-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/hello.txt', 'content': 'Deep Agent가 만든 파일입니다.'}, 'id': 'call_WMINjFcXpq7BemF9m0sz5AEV', 'type': 'tool_call'}], inv

In [20]:
result = agent.invoke({"messages":HumanMessage(content="'./agent_workspace/sample_data'에 있는 엑셀데이터를 활용한 html 대시보드를 수정해줘. A열은 ticker, B열은 종목명, H열은 거래대금에 관한 데이터가 있어. 각 파일명은 투자주체를 담고 있고 내가 원하는 것은 특정 종목에 대한 투자주체별 정보를 Tracking하고 싶은 목적이야")})
result

{'messages': [HumanMessage(content="'./agent_workspace/sample_data'에 있는 엑셀데이터를 활용한 html 대시보드를 수정해줘. A열은 ticker, B열은 종목명, H열은 거래대금에 관한 데이터가 있어. 각 파일명은 투자주체를 담고 있고 내가 원하는 것은 특정 종목에 대한 투자주체별 정보를 Tracking하고 싶은 목적이야", additional_kwargs={}, response_metadata={}, id='60e13de3-d362-4d54-b48b-9e629a826bfb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 5566, 'total_tokens': 5672, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 4736}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Ds0S8aaz40R8KuhT1ykWBLpNx6sSA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed966-5a33-7c53-bf27-bb87df7da36c-0', tool_calls=[{'name': 'write_todos', 'args': {'to

## [1-3] : AGENTS.md — 항상 로드되는 지침

**AGENTS.md**는 에이전트에게 **항상 적용**되어야 하는 규칙, 컨벤션, 컨텍스트 정보를 담는 마크다운 파일입니다.

### 특징
- 에이전트가 시작할 때 **항상 로드** (Always-on Context)
- `<agent_memory>` 태그로 시스템 프롬프트에 주입
- 여러 소스를 배열로 지정 가능
- 에이전트가 `edit_file` 도구로 **스스로 업데이트** 가능

### AGENTS.md 저장 위치

| 범위 | 위치 | 용도 |
|------|------|------|
| 글로벌 | `~/.deepagents/<agent>/memories/` | 모든 프로젝트에 적용되는 사용자 선호도 |
| 프로젝트 | `.deepagents/AGENTS.md` | 현재 프로젝트의 컨벤션, 아키텍처 규칙 |

### AGENTS.md 예시

```markdown
# Project Guidelines

## 코딩 컨벤션
- Python 3.12+ 사용
- Black 포맷터, isort 적용
- 타입 힌트 필수
- 독스트링은 Google 스타일

## 아키텍처 원칙
- Clean Architecture 기반
- 의존성 주입 패턴 사용
- 환경변수는 .env 파일로 관리
```

In [12]:
# [7-7] : AGENTS.md를 memory 파라미터로 로드하는 에이전트 생성
# [주의] StateBackend는 휘발성 — 스레드 종료 시 데이터가 사라짐
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
import tempfile

# 스킬 저장 폴더 생성
skills_dir = Path("skills").resolve()
skills_dir.mkdir(parents=True, exist_ok=True)
skills_dir

WindowsPath('C:/Users/User/hynix_agent/skills')

In [13]:
from pathlib import Path

# 루트 노드
project_root = Path.cwd().resolve()
project_root

WindowsPath('C:/Users/User/hynix_agent')

In [21]:
agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(
        root_dir=str(project_root),
        virtual_mode=True
    ),
    memory=['./AGENTS.md'],
    skills=['./skills/']
)

In [26]:
result = agent.invoke({"messages":[HumanMessage(content="파이썬으로 오목 게임 코드 작성해줘. 그리고 작성 완료된 파이썬 코드에 대한 구동 방식 설명을 해줘")]})

print(result["messages"][-1].content)

아래 코드로 오목 게임을 실행할 수 있습니다.  
현재 파일 기준으로는 `/omok_game.py`에 구현되어 있습니다.

```python
BOARD_SIZE = 15
EMPTY = "."
BLACK = "B"
WHITE = "W"


def create_board():
    return [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]


def print_board(board):
    print("   " + " ".join(f"{i+1:2}" for i in range(BOARD_SIZE)))
    for y, row in enumerate(board):
        print(f"{y+1:2} " + " ".join(f" {cell}" for cell in row))


def in_bounds(x, y):
    return 0 <= x < BOARD_SIZE and 0 <= y < BOARD_SIZE


def check_five(board, x, y, stone):
    directions = [(1, 0), (0, 1), (1, 1), (1, -1)]
    for dx, dy in directions:
        count = 1
        nx, ny = x + dx, y + dy
        while in_bounds(nx, ny) and board[ny][nx] == stone:
            count += 1
            nx += dx
            ny += dy

        nx, ny = x - dx, y - dy
        while in_bounds(nx, ny) and board[ny][nx] == stone:
            count += 1
            nx -= dx
            ny -= dy

        if count >= 5:
            return True

In [27]:
def show_skill_reads(result):
    found = False

    for msg in result["messages"]:
        tool_calls = getattr(msg, "tool_calls", None)

        if not tool_calls:
            continue

        for tool_call in tool_calls:
            name = tool_call.get("name")
            args = tool_call.get("args", {})

            if name == "read_file":
                path = args.get("file_path") or args.get("path") or ""

                if "/skills/" in path and path.endswith("SKILL.md"):
                    print("호출된 스킬 파일:", path)
                    found = True

    if not found:
        print("이번 실행에서는 SKILL.md를 직접 읽은 흔적이 없습니다.")

show_skill_reads(result)

이번 실행에서는 SKILL.md를 직접 읽은 흔적이 없습니다.


## [1-4] : SKILL.md — 온디맨드 스킬 정의

**SKILL.md**는 에이전트에게 **특정 태스크 전문 지식**을 부여하는 선언적 파일입니다.

### AGENTS.md vs SKILL.md

| 비교 | AGENTS.md | SKILL.md |
|------|-----------|----------|
| **로딩** | 항상 (Always-on) | 필요 시 (On-demand) |
| **용도** | 프로젝트 규칙, 컨벤션 | 특정 태스크 워크플로 |
| **크기** | 간결할수록 좋음 | 대용량 가능 (10MB 제한) |
| **토큰 효율** | 항상 소비 | Progressive Disclosure로 절약 |
| **업데이트** | 에이전트가 edit_file로 수정 가능 | 보통 정적 |

### Progressive Disclosure (점진적 공개) 패턴

스킬은 한 번에 전부 로드하지 않습니다:

```
1단계: 프론트매터만 로드        2단계: 에이전트가 관련성 판단    3단계: 전체 내용 로드
┌─────────────────┐          ┌─────────────────┐           ┌─────────────────┐
│ name: rag-agent │    →     │ "이 요청은 RAG   │     →     │ # RAG Agent     │
│ description: .. │          │  스킬이 필요해"  │           │ ## 사용 시기     │
│ allowed-tools:  │          │                 │           │ ## 워크플로      │
│   retrieve      │          │                 │           │ ## 안전 규칙     │
└─────────────────┘          └─────────────────┘           └─────────────────┘
   ~50 tokens                                                ~500+ tokens
```

> **[핵심]** Progressive Disclosure 덕분에 스킬이 100개여도 초기 토큰 비용은 프론트매터 합산분뿐입니다.

### SKILL.md 구조

```yaml
---
name: my-skill               # 스킬 식별자 (최대 64자, 소문자+하이픈)
description: >               # 설명 (최대 1024자) — 매칭에 사용
  이 스킬이 무엇을 하는지 설명합니다.
license: MIT
compatibility: Python 3.12+
metadata:
  category: domain
  difficulty: intermediate
allowed-tools: tool1 tool2   # 스킬이 사용할 수 있는 도구 목록
---

# 스킬 본문 (마크다운)

## 사용 시기
- 조건 1
- 조건 2

## 워크플로
1. 단계 1
2. 단계 2

## 안전 규칙
- 규칙 1
```

In [9]:
# [7-8] : Progressive Disclosure 시뮬레이션 — 토큰 절약 효과
# [핵심] SKILL.md는 Agent 행동을 선언적으로 정의하는 명세서
# [주의] 실행 전 API Key 설정을 확인하세요
skill_frontmatter_examples = [
    {
        "name": "rag-agent",
        "description": "RAG(Retrieval-Augmented Generation) 에이전트 구축 가이드",
        "allowed_tools": ["retrieve"],
        "full_content_tokens": 520,
    },
    {
        "name": "sql-agent",
        "description": "자연어 SQL 질의 에이전트 가이드",
        "allowed_tools": ["sql_db_list_tables", "sql_db_schema", "sql_db_query", "sql_db_query_checker"],
        "full_content_tokens": 480,
    },
    {
        "name": "code-review",
        "description": "코드 품질 검토 및 개선 가이드",
        "allowed_tools": ["read_file", "grep", "write_file"],
        "full_content_tokens": 650,
    },
]

print("=== Progressive Disclosure 시뮬레이션 ===")
print()

# 1단계: 프론트매터만 (약 50 tokens each)
frontmatter_tokens = len(skill_frontmatter_examples) * 50
full_tokens = sum(s["full_content_tokens"] for s in skill_frontmatter_examples)

print(f"등록된 스킬: {len(skill_frontmatter_examples)}개")
print(f"프론트매터만 로드 시: ~{frontmatter_tokens} tokens")
print(f"전체 로드 시:        ~{full_tokens} tokens")
print(f"절약:               ~{full_tokens - frontmatter_tokens} tokens ({(1 - frontmatter_tokens/full_tokens)*100:.0f}%)")
print()

for skill in skill_frontmatter_examples:
    print(f"  [{skill['name']}]")
    print(f"    설명: {skill['description']}")
    print(f"    도구: {', '.join(skill['allowed_tools'])}")
    print(f"    전체 로드 시: ~{skill['full_content_tokens']} tokens")

=== Progressive Disclosure 시뮬레이션 ===

등록된 스킬: 3개
프론트매터만 로드 시: ~150 tokens
전체 로드 시:        ~1650 tokens
절약:               ~1500 tokens (91%)

  [rag-agent]
    설명: RAG(Retrieval-Augmented Generation) 에이전트 구축 가이드
    도구: retrieve
    전체 로드 시: ~520 tokens
  [sql-agent]
    설명: 자연어 SQL 질의 에이전트 가이드
    도구: sql_db_list_tables, sql_db_schema, sql_db_query, sql_db_query_checker
    전체 로드 시: ~480 tokens
  [code-review]
    설명: 코드 품질 검토 및 개선 가이드
    도구: read_file, grep, write_file
    전체 로드 시: ~650 tokens


## [1-5] : SKILL.md 직접 작성 실습

이제 실제 SKILL.md를 작성해 봅니다.  
아래 두 가지 에이전트 스킬을 직접 정의합니다:

1. **RAG Agent 스킬** — 문서 기반 질의응답
2. **SQL Agent 스킬** — 자연어 데이터베이스 질의

In [10]:
# [7-9] : RAG Agent SKILL.md 작성
# [핵심] SKILL.md는 Agent 행동을 선언적으로 정의하는 명세서
# [주의] SKILL.md의 tools 목록은 허용 도구만 명시 — 미등록 도구는 실행 불가
rag_skill_md = '''\
---
name: rag-agent
description: >
  RAG(Retrieval-Augmented Generation) 에이전트 구축 가이드.
  벡터 스토어 검색, 문서 청킹, content_and_artifact 패턴을 다룹니다.
license: MIT
compatibility: Python 3.12+
metadata:
  category: rag
  difficulty: intermediate
allowed-tools: retrieve
---

# RAG Agent 스킬

## 사용 시기
- 사용자가 문서 기반 질의응답 시스템을 구축할 때
- 벡터 스토어에서 관련 문서를 검색하고 답변을 생성할 때
- RAG 파이프라인의 검색 품질을 개선할 때

## 워크플로
1. **문서 준비**: 원본 문서를 RecursiveCharacterTextSplitter로 청킹
2. **임베딩 & 저장**: OpenAIEmbeddings -> InMemoryVectorStore
3. **검색 도구 정의**: @tool(response_format="content_and_artifact") 패턴
4. **에이전트 생성**: create_deep_agent(tools=[retrieve])
5. **질의 & 검증**: 단순 질의 -> 비교 질의 -> 출처 확인

## 핵심 패턴: content_and_artifact
```python
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    results = vectorstore.similarity_search(query, k=3)
    content = "\\n\\n".join(d.page_content for d in results)
    return content, results  # (요약 텍스트, 원본 객체)
```

## 안전 규칙
- 검색된 문서에 없는 내용은 추측하지 않는다
- 출처(문서 제목, 페이지)를 반드시 명시한다
- 검색 결과가 부족하면 사용자에게 알린다
'''

print("=== RAG Agent SKILL.md ===")
print(rag_skill_md)
print(f"총 길이: {len(rag_skill_md)} 문자")

=== RAG Agent SKILL.md ===
---
name: rag-agent
description: >
  RAG(Retrieval-Augmented Generation) 에이전트 구축 가이드.
  벡터 스토어 검색, 문서 청킹, content_and_artifact 패턴을 다룹니다.
license: MIT
compatibility: Python 3.12+
metadata:
  category: rag
  difficulty: intermediate
allowed-tools: retrieve
---

# RAG Agent 스킬

## 사용 시기
- 사용자가 문서 기반 질의응답 시스템을 구축할 때
- 벡터 스토어에서 관련 문서를 검색하고 답변을 생성할 때
- RAG 파이프라인의 검색 품질을 개선할 때

## 워크플로
1. **문서 준비**: 원본 문서를 RecursiveCharacterTextSplitter로 청킹
2. **임베딩 & 저장**: OpenAIEmbeddings -> InMemoryVectorStore
3. **검색 도구 정의**: @tool(response_format="content_and_artifact") 패턴
4. **에이전트 생성**: create_deep_agent(tools=[retrieve])
5. **질의 & 검증**: 단순 질의 -> 비교 질의 -> 출처 확인

## 핵심 패턴: content_and_artifact
```python
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    results = vectorstore.similarity_search(query, k=3)
    content = "\n\n".join(d.page_content for d in results)
    return content, results  # (요약 텍스트, 원본 객체)
```

## 안전 규칙
- 검색된 문서에 없는 내용은 추측하지 않는다
- 출처

In [11]:
# [7-10] : SQL Agent SKILL.md 작성
# [핵심] SKILL.md는 Agent 행동을 선언적으로 정의하는 명세서
# [주의] SKILL.md의 tools 목록은 허용 도구만 명시 — 미등록 도구는 실행 불가
sql_skill_md = '''\
---
name: sql-agent
description: >
  자연어 SQL 질의 에이전트 가이드.
  SQLDatabaseToolkit 활용, READ-ONLY 안전 규칙, HITL 승인 패턴을 다룹니다.
license: MIT
compatibility: Python 3.12+
metadata:
  category: sql
  difficulty: intermediate
allowed-tools: sql_db_list_tables sql_db_schema sql_db_query sql_db_query_checker
---

# SQL Agent 스킬

## 사용 시기
- 사용자가 자연어로 데이터베이스를 질의할 때
- SQL 쿼리 생성 및 실행을 자동화할 때
- 데이터베이스 스키마를 탐색하고 분석할 때

## 워크플로: query-writing
1. sql_db_list_tables -- 사용 가능한 테이블 목록 확인
2. sql_db_schema -- 관련 테이블의 DDL 조회
3. SQL 쿼리 작성 -- 스키마 기반 정확한 쿼리 생성
4. sql_db_query_checker -- 쿼리 문법 및 안전성 검증
5. sql_db_query -- 검증된 쿼리 실행

## 안전 규칙 (READ-ONLY)
- SELECT만 허용: INSERT, UPDATE, DELETE, DROP, ALTER 금지
- LIMIT 필수: 항상 LIMIT 10 사용
- 스키마 확인 필수: 쿼리 작성 전 반드시 테이블 스키마 확인
- HITL 승인: sql_db_query 실행 전 사람의 승인 획득 (프로덕션)

## HITL 패턴
```python
middleware=[
    HumanInTheLoopMiddleware(interrupt_on={"sql_db_query": True}),
]
```
'''

print("=== SQL Agent SKILL.md ===")
print(sql_skill_md)
print(f"총 길이: {len(sql_skill_md)} 문자")

=== SQL Agent SKILL.md ===
---
name: sql-agent
description: >
  자연어 SQL 질의 에이전트 가이드.
  SQLDatabaseToolkit 활용, READ-ONLY 안전 규칙, HITL 승인 패턴을 다룹니다.
license: MIT
compatibility: Python 3.12+
metadata:
  category: sql
  difficulty: intermediate
allowed-tools: sql_db_list_tables sql_db_schema sql_db_query sql_db_query_checker
---

# SQL Agent 스킬

## 사용 시기
- 사용자가 자연어로 데이터베이스를 질의할 때
- SQL 쿼리 생성 및 실행을 자동화할 때
- 데이터베이스 스키마를 탐색하고 분석할 때

## 워크플로: query-writing
1. sql_db_list_tables -- 사용 가능한 테이블 목록 확인
2. sql_db_schema -- 관련 테이블의 DDL 조회
3. SQL 쿼리 작성 -- 스키마 기반 정확한 쿼리 생성
4. sql_db_query_checker -- 쿼리 문법 및 안전성 검증
5. sql_db_query -- 검증된 쿼리 실행

## 안전 규칙 (READ-ONLY)
- SELECT만 허용: INSERT, UPDATE, DELETE, DROP, ALTER 금지
- LIMIT 필수: 항상 LIMIT 10 사용
- 스키마 확인 필수: 쿼리 작성 전 반드시 테이블 스키마 확인
- HITL 승인: sql_db_query 실행 전 사람의 승인 획득 (프로덕션)

## HITL 패턴
```python
middleware=[
    HumanInTheLoopMiddleware(interrupt_on={"sql_db_query": True}),
]
```

총 길이: 897 문자


In [12]:
# [7-11] : SKILL.md를 사용하는 에이전트 생성 — skills 파라미터
# [주의] StateBackend는 휘발성 — 스레드 종료 시 데이터가 사라짐
skilled_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 시니어 개발자입니다. 사용 가능한 스킬을 활용하여 작업을 수행하세요.",
    backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    skills=["/skills/"],  # [핵심] 스킬 소스 디렉토리 — SKILL.md 파일 자동 탐색
)

print("스킬 에이전트 생성 완료")
print()
print("스킬 우선순위 (여러 소스 지정 시):")
print('  skills=["/skills/base/", "/skills/user/", "/skills/project/"]')
print("  -> 나중에 나온 소스가 우선 (last wins)")
print("  -> 같은 이름의 스킬이 있으면 마지막 소스의 스킬이 사용됨")

스킬 에이전트 생성 완료

스킬 우선순위 (여러 소스 지정 시):
  skills=["/skills/base/", "/skills/user/", "/skills/project/"]
  -> 나중에 나온 소스가 우선 (last wins)
  -> 같은 이름의 스킬이 있으면 마지막 소스의 스킬이 사용됨


In [13]:
from pathlib import Path
from textwrap import dedent

from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend


# ---------------------------------------------------------
# model은 이미 로드되어 있다고 가정
# 예:
# model = init_chat_model(...)
# ---------------------------------------------------------


# 1. Skill 디렉터리 준비
PROJECT_ROOT = Path("./deepagent_skill_demo").resolve()
SKILLS_ROOT = PROJECT_ROOT / "skills"


def create_skill(
    name: str,
    description: str,
    instructions: str,
) -> None:
    """skills/<name>/SKILL.md 파일을 생성한다."""
    skill_dir = SKILLS_ROOT / name
    skill_dir.mkdir(parents=True, exist_ok=True)

    skill_content = dedent(
        f"""
        ---
        name: {name}
        description: >-
          {description}
        ---

        # {name}

        {instructions.strip()}
        """
    ).strip()

    (skill_dir / "SKILL.md").write_text(
        skill_content + "\n",
        encoding="utf-8",
    )


# ---------------------------------------------------------
# 2. Skill 두 개 생성
# ---------------------------------------------------------

create_skill(
    name="three-line-summary",
    description=(
        "사용자가 글, 문서 또는 문장을 세 줄로 요약해 달라고 요청할 때 "
        "사용하는 한국어 요약 스킬이다."
    ),
    instructions="""
    ## 수행 방법

    1. 입력 내용에서 가장 중요한 정보만 추출한다.
    2. 중복 내용과 부가 설명은 제거한다.
    3. 결과를 정확히 세 개의 글머리표로 작성한다.

    ## 출력 규칙

    - 첫 줄에 `[three-line-summary 스킬 사용]`을 출력한다.
    - 그다음 정확히 세 개의 글머리표를 출력한다.
    - 각 항목은 한 문장으로 작성한다.
    - 새로운 사실을 임의로 추가하지 않는다.
    """,
)

create_skill(
    name="polite-email",
    description=(
        "사용자가 메모나 문장을 정중한 한국어 업무 이메일로 작성하거나 "
        "변환해 달라고 요청할 때 사용하는 이메일 작성 스킬이다."
    ),
    instructions="""
    ## 수행 방법

    사용자의 내용을 다음 구조의 정중한 업무 이메일로 변환한다.

    1. 제목
    2. 인사말
    3. 요청 또는 전달 내용
    4. 감사 및 마무리 인사

    ## 출력 규칙

    - 첫 줄에 `[polite-email 스킬 사용]`을 출력한다.
    - 제목은 `제목:`으로 시작한다.
    - 과도하게 장황한 표현은 사용하지 않는다.
    - 사용자가 제공하지 않은 이름이나 회사명은 만들지 않는다.
    """,
)


# ---------------------------------------------------------
# 3. 파일시스템 Backend 및 Deep Agent 생성
# ---------------------------------------------------------

backend = FilesystemBackend(
    root_dir=PROJECT_ROOT,
    virtual_mode=True,
)

agent = create_deep_agent(
    model=model,
    backend=backend,

    # PROJECT_ROOT를 가상 루트 "/"로 보기 때문에
    # 실제 ./deepagent_skill_demo/skills 디렉터리를 의미한다.
    skills=["/skills/"],

    # 데모에서 Skill 로드가 눈에 잘 보이도록 명시
    system_prompt=dedent(
        """
        당신은 한국어 업무 보조 에이전트다.

        사용자의 요청과 일치하는 Skill이 있다면 반드시 해당 Skill의
        SKILL.md 파일을 read_file 도구로 먼저 읽은 후 답변하라.

        Skill의 지침과 출력 형식을 정확히 준수하라.
        """
    ).strip(),
)


# ---------------------------------------------------------
# 4. 호출된 Skill을 출력하는 실행 함수
# ---------------------------------------------------------

def run_agent(prompt: str) -> None:
    print("\n" + "=" * 70)
    print(f"사용자: {prompt}")

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
        }
    )

    messages = result["messages"]

    # AI가 호출한 read_file 도구 중 SKILL.md 로드를 찾아 출력
    for message in messages:
        tool_calls = getattr(message, "tool_calls", None) or []

        for tool_call in tool_calls:
            tool_name = tool_call.get("name")
            tool_args = tool_call.get("args", {})

            if tool_name != "read_file":
                continue

            file_path = (
                tool_args.get("file_path")
                or tool_args.get("path")
                or ""
            )

            if "skills" in file_path and "SKILL.md" in file_path:
                print(f"\n[호출된 Skill] {file_path}")

    # 마지막 AI 답변 찾기
    final_answer = ""

    for message in reversed(messages):
        content = getattr(message, "content", None)
        message_type = getattr(message, "type", None)

        if message_type == "ai" and content:
            final_answer = content
            break

    print("\n에이전트 응답:")
    print(final_answer)


# ---------------------------------------------------------
# 5. Skill 두 개 각각 호출
# ---------------------------------------------------------

run_agent(
    """
    다음 내용을 세 줄로 요약해줘.

    신규 주문 시스템은 다음 달부터 적용된다.
    기존 사용자는 별도의 가입 절차 없이 이용할 수 있다.
    관리자 교육은 이번 달 마지막 주에 진행한다.
    시스템 적용 전에 데이터 백업이 필요하다.
    """
)

run_agent(
    """
    아래 메모를 정중한 업무 이메일로 바꿔줘.

    내일까지 지난달 매출 자료 보내주세요.
    누락된 지점 데이터도 확인 부탁드립니다.
    """
)

Skipping /skills/polite-email/SKILL.md: no valid YAML frontmatter found
Skill at /skills/polite-email/SKILL.md failed metadata parse or name validation; skipping
Skipping /skills/three-line-summary/SKILL.md: no valid YAML frontmatter found
Skill at /skills/three-line-summary/SKILL.md failed metadata parse or name validation; skipping



사용자: 
    다음 내용을 세 줄로 요약해줘.

    신규 주문 시스템은 다음 달부터 적용된다.
    기존 사용자는 별도의 가입 절차 없이 이용할 수 있다.
    관리자 교육은 이번 달 마지막 주에 진행한다.
    시스템 적용 전에 데이터 백업이 필요하다.
    


Skipping /skills/polite-email/SKILL.md: no valid YAML frontmatter found
Skill at /skills/polite-email/SKILL.md failed metadata parse or name validation; skipping
Skipping /skills/three-line-summary/SKILL.md: no valid YAML frontmatter found
Skill at /skills/three-line-summary/SKILL.md failed metadata parse or name validation; skipping



에이전트 응답:
신규 주문 시스템이 다음 달부터 적용됩니다.  
기존 사용자는 별도 가입 없이 이용 가능하며, 관리자 교육은 이번 달 마지막 주에 진행됩니다.  
시스템 적용 전 데이터 백업이 필요합니다.

사용자: 
    아래 메모를 정중한 업무 이메일로 바꿔줘.

    내일까지 지난달 매출 자료 보내주세요.
    누락된 지점 데이터도 확인 부탁드립니다.
    

에이전트 응답:
다음과 같이 정중한 업무 이메일로 변경해 드립니다.

---

제목: 지난달 매출 자료 및 누락 지점 데이터 요청드립니다.

안녕하세요,  
귀하의 업무에 감사드립니다.

혹시 가능하시다면, 내일까지 지난달 매출 자료를 보내주실 수 있을까요?  
아울러 누락된 지점 데이터가 있는지도 함께 확인 부탁드립니다.

바쁘신 와중에 번거롭게 해드려 죄송하지만, 확인 후 회신 주시면 감사하겠습니다.

감사합니다.  
[작성자 이름 드림]


---
# Part 2 : 스토리지 백엔드 + 샌드박스 + 배포 설계

## [2-1] : 스토리지 백엔드 아키텍처

Deep Agents의 파일 도구(`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`)는  
모두 **백엔드(Backend)**를 통해 동작합니다.  
백엔드는 에이전트가 파일을 읽고 쓰는 **스토리지 계층을 추상화**합니다.

### BackendProtocol 인터페이스

모든 백엔드는 동일한 프로토콜을 구현합니다:

| 메서드 | 설명 |
|--------|------|
| `ls_info(path)` | 디렉토리 내용 목록 |
| `read(file_path, offset, limit)` | 파일 읽기 (줄 번호 포함) |
| `write(file_path, content)` | 새 파일 생성 |
| `edit(file_path, old_string, new_string)` | 텍스트 교체 |
| `grep_raw(pattern, path, glob)` | 패턴 기반 파일 내용 검색 |
| `glob_info(pattern, path)` | 글로브 패턴으로 파일 검색 |

> **[핵심]** 백엔드만 바꾸면 에이전트의 파일 도구가 동작하는 **저장소를 플러그인처럼 교체**할 수 있습니다.

## [2-2] : 4가지 백엔드 유형

| 백엔드 | 저장 위치 | 영속성 | 사용 시나리오 |
|--------|----------|--------|-------------|
| `StateBackend` | 에이전트 상태 (메모리) | 스레드 내 | 임시 작업, 스크래치패드 **(기본값)** |
| `FilesystemBackend` | 로컬 디스크 | 영구 | 로컬 파일 접근, 코딩 에이전트 |
| `StoreBackend` | LangGraph Store | 크로스 스레드 | 장기 메모리, 사용자 선호도 |
| `CompositeBackend` | 경로별 라우팅 | 혼합 | 메모리 + 임시 파일 병용 |

### StateBackend (에페메럴, 기본값)
- `create_deep_agent()`에서 `backend`를 지정하지 않으면 자동 사용
- 대화 스레드 내에서만 파일 유지, 스레드 종료 시 소멸

### FilesystemBackend (로컬 디스크)
- `root_dir`로 접근 가능한 루트 디렉토리 지정
- `virtual_mode=True`로 경로 제한 (`..`, `~` 등 차단)

### StoreBackend (크로스 스레드)
- LangGraph의 `BaseStore`를 활용, 스레드 간 파일 공유
- Redis, PostgreSQL 등 다양한 스토어 지원

### CompositeBackend (경로별 라우팅)
- 서로 다른 경로를 서로 다른 백엔드로 라우팅
- 가장 일반적: `/memories/*`는 영속 저장, 나머지는 에페메럴

In [12]:
# [7-12] : 4가지 백엔드 임포트 및 비교
# [핵심] CompositeBackend는 경로 기반으로 여러 백엔드를 라우팅
# [주의] StateBackend는 휘발성 — 스레드 종료 시 데이터가 사라짐
from deepagents.backends import (
    StateBackend,
    FilesystemBackend,
    StoreBackend,
    CompositeBackend,
)

backend_comparison = [
    {"name": "StateBackend",      "persistence": "스레드 내",     "storage": "에이전트 상태",    "use_case": "임시 작업 (기본값)"},
    {"name": "FilesystemBackend", "persistence": "영구",         "storage": "로컬 디스크",      "use_case": "코딩 에이전트"},
    {"name": "StoreBackend",      "persistence": "크로스 스레드", "storage": "LangGraph Store",  "use_case": "장기 메모리"},
    {"name": "CompositeBackend",  "persistence": "혼합",         "storage": "경로별 라우팅",    "use_case": "메모리+임시 병용"},
]

print("=== 백엔드 유형 비교 ===")
print(f"{'백엔드':<22s} {'영속성':<14s} {'저장소':<18s} {'사용 시나리오'}")
print("-" * 80)
for b in backend_comparison:
    print(f"{b['name']:<22s} {b['persistence']:<14s} {b['storage']:<18s} {b['use_case']}")

print("\n모든 백엔드 클래스를 성공적으로 임포트했습니다!")

=== 백엔드 유형 비교 ===
백엔드                    영속성            저장소                사용 시나리오
--------------------------------------------------------------------------------
StateBackend           스레드 내          에이전트 상태            임시 작업 (기본값)
FilesystemBackend      영구             로컬 디스크             코딩 에이전트
StoreBackend           크로스 스레드        LangGraph Store    장기 메모리
CompositeBackend       혼합             경로별 라우팅            메모리+임시 병용

모든 백엔드 클래스를 성공적으로 임포트했습니다!


## [2-3] : CompositeBackend로 메모리 영속성 설계

`CompositeBackend`는 가장 실용적인 패턴입니다.  
**경로 접두사(prefix)**에 따라 서로 다른 백엔드로 요청을 라우팅합니다.

```
                    CompositeBackend
                   ┌───────────────────┐
  /memories/*  --> │  StoreBackend     │  <-- 크로스 스레드 영속
                   ├───────────────────┤
  /workspace/* --> │ FilesystemBackend │  <-- 로컬 디스크
                   ├───────────────────┤
  그 외 경로   --> │  StateBackend     │  <-- 에페메럴 (기본)
                   └───────────────────┘
```

In [13]:
# [7-13] : CompositeBackend 에이전트 생성 — 경로별 라우팅
# [주의] StateBackend는 휘발성 — 스레드 종료 시 데이터가 사라짐
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

store = InMemoryStore()          # 개발용 (프로덕션: PostgresStore)
checkpointer = MemorySaver()     # 에이전트 상태 유지


# [핵심] CompositeBackend 팩토리 — /memories/만 영속, 나머지는 에페메럴
def memory_backend_factory(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),           # 기본: 에페메럴
        routes={
            "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장
        },
    )


composite_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 개인 어시스턴트입니다.
사용자가 기억해 달라고 하는 정보는 /memories/ 폴더에 저장하세요.
이전에 저장한 메모리가 있으면 참고하여 응답하세요.
한국어로 응답하세요.""",
    backend=memory_backend_factory,
    store=store,
    checkpointer=checkpointer,
)

print("CompositeBackend 에이전트 생성 완료")
print("  /memories/* -> StoreBackend (크로스 스레드 영속)")
print("  그 외      -> StateBackend (에페메럴)")

CompositeBackend 에이전트 생성 완료
  /memories/* -> StoreBackend (크로스 스레드 영속)
  그 외      -> StateBackend (에페메럴)


In [14]:
# [7-14] : 크로스 스레드 메모리 공유 테스트
# 스레드 1에서 선호도 저장
# [핵심] 이 코드는 하네스 엔지니어링의 핵심 구성 요소를 구현
# [주의] 실행 전 API Key 설정을 확인하세요
config_t1 = {"configurable": {"thread_id": "thread-1"}}

result1 = composite_agent.invoke(
    {"messages": [{"role": "user", "content":
        "다음 정보를 /memories/preferences.txt에 저장해 주세요: "
        "선호 언어=Python, 에디터=VS Code, 포맷터=Black"}]},
    config={**config_t1, **lf_config},
)
print("[스레드 1] 저장 결과:")
print(result1["messages"][-1].content)
print()

# 스레드 2에서 저장된 메모리 읽기
config_t2 = {"configurable": {"thread_id": "thread-2"}}

result2 = composite_agent.invoke(
    {"messages": [{"role": "user", "content":
        "/memories/preferences.txt를 읽어서 내 선호도를 알려 주세요."}]},
    config={**config_t2, **lf_config},
)
print("[스레드 2] 메모리 읽기 결과:")
print(result2["messages"][-1].content)

C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기본: 에페메럴
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:15: DeprecationWarning: Passing `runtime` to StoreBackend is deprecated and will be removed in v0.7. StoreBackend now obtains store and context via `get_store()` / `get_runtime()`. Simply use `StoreBackend()` or `StoreBackend(store=my_store)` instead.
  "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장


C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기본: 에페메럴
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:15: DeprecationWarning: Passing `runtime` to StoreBackend is deprecated and will be removed in v0.7. StoreBackend now obtains store and context via `get_store()` / `get_runtime()`. Simply use `StoreBackend()` or `StoreBackend(store=my_store)` instead.
  "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기

[스레드 1] 저장 결과:
선호 언어=Python, 에디터=VS Code, 포맷터=Black 정보를 /memories/preferences.txt에 저장했습니다.



C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기본: 에페메럴
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:15: DeprecationWarning: Passing `runtime` to StoreBackend is deprecated and will be removed in v0.7. StoreBackend now obtains store and context via `get_store()` / `get_runtime()`. Simply use `StoreBackend()` or `StoreBackend(store=my_store)` instead.
  "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장


C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기본: 에페메럴
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:15: DeprecationWarning: Passing `runtime` to StoreBackend is deprecated and will be removed in v0.7. StoreBackend now obtains store and context via `get_store()` / `get_runtime()`. Simply use `StoreBackend()` or `StoreBackend(store=my_store)` instead.
  "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기

C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기본: 에페메럴
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:15: DeprecationWarning: Passing `runtime` to StoreBackend is deprecated and will be removed in v0.7. StoreBackend now obtains store and context via `get_store()` / `get_runtime()`. Simply use `StoreBackend()` or `StoreBackend(store=my_store)` instead.
  "/memories/": StoreBackend(runtime),  # /memories/* -> 영속 저장
C:\Users\HEESU\AppData\Local\Temp\ipykernel_20428\694171000.py:13: DeprecationWarning: Passing `runtime` to StateBackend is deprecated and will be removed in v0.7. StateBackend now reads and writes state via `get_config()`. Simply use `StateBackend()` instead.
  default=StateBackend(runtime),           # 기

[스레드 2] 메모리 읽기 결과:
당신의 선호도는 다음과 같습니다:
- 선호 언어: Python
- 에디터: VS Code
- 포맷터: Black


## [2-4] : 샌드박스 보안 패턴

**샌드박스**는 AI 에이전트가 코드를 실행할 수 있는 **격리된 실행 환경**입니다.

### 왜 격리가 필요한가?

| 위험 | 격리 없을 때 | 샌드박스 사용 시 |
|------|------------|----------------|
| 파일 시스템 | 호스트 파일 변경/삭제 가능 | 격리된 파일시스템만 접근 |
| 네트워크 | 무제한 외부 통신 | 제한된 네트워크 접근 |
| 자격 증명 | 환경 변수 유출 가능 | 시크릿 격리 |
| 시스템 | 호스트 OS에 영향 | 호스트 시스템 보호 |

### 두 가지 아키텍처 패턴

| 패턴 | 설명 | 권장 여부 |
|------|------|-----------|
| **Agent-in-Sandbox** | 에이전트가 샌드박스 **내부**에서 실행 | 자격 증명 노출 위험 |
| **Sandbox-as-Tool** | 에이전트가 **외부**에서 샌드박스 API 호출 | **권장** |

> **[핵심]** Sandbox-as-Tool 패턴에서는 시크릿이 에이전트(호스트) 측에만 존재하므로  
> 샌드박스 내부에서 자격 증명이 유출될 위험이 없습니다.

### 샌드박스 프로바이더

| 프로바이더 | 특징 | 적합한 용도 |
|-----------|------|------------|
| **Modal** | GPU 지원, ML 워크로드 | AI/ML 작업, 데이터 처리 |
| **Daytona** | TypeScript/Python, 빠른 콜드 스타트 | 웹 개발, 빠른 반복 |
| **Runloop** | 일회용 devbox, 격리 실행 | 코드 테스트, 일회성 작업 |

### HITL(Human-in-the-Loop) 승인

```python
agent = create_deep_agent(
    model="gpt-4.1",
    backend=ModalSandbox(image="python:3.12-slim"),
    interrupt_on={"execute": True},  # 코드 실행 전 사람 승인 필요
)
```

In [15]:
# [7-15] : 샌드박스 아키텍처 패턴 비교 (참고용)
# [핵심] Sandbox-as-Tool 패턴으로 코드 실행을 격리된 환경에서 수행
print("=== 패턴 1: Agent-in-Sandbox ===")
print("  [샌드박스]")
print("    |-- 에이전트 (내부 실행)")
print("    |-- 파일시스템")
print("    |-- 코드 실행")
print("    <---> 네트워크 프로토콜 <---> 외부 시스템")
print()
print("=== 패턴 2: Sandbox-as-Tool (권장) ===")
print("  [호스트]")
print("    |-- 에이전트 (외부 실행)")
print("    |-- 자격 증명 관리")
print("    |-- API 호출 --> [샌드박스]")
print("                       |-- 파일시스템")
print("                       |-- 코드 실행")
print()

# [주의] 보안 가이드라인
print("=== 보안 가이드라인 ===")
security_rules = [
    "1. 자격 증명은 외부 도구에서만 관리 (샌드박스 안에 시크릿 넣지 마세요)",
    "2. Human-in-the-Loop -- 민감한 작업에 사람 승인 요구",
    "3. 네트워크 접근 차단 -- 불필요한 아웃바운드 연결 차단",
    "4. 아웃바운드 모니터링 -- 예기치 않은 외부 연결 감시",
    "5. 출력 검토 -- 샌드박스 출력을 적용 전 검토",
]
for rule in security_rules:
    print(f"  {rule}")

=== 패턴 1: Agent-in-Sandbox ===
  [샌드박스]
    |-- 에이전트 (내부 실행)
    |-- 파일시스템
    |-- 코드 실행
    <---> 네트워크 프로토콜 <---> 외부 시스템

=== 패턴 2: Sandbox-as-Tool (권장) ===
  [호스트]
    |-- 에이전트 (외부 실행)
    |-- 자격 증명 관리
    |-- API 호출 --> [샌드박스]
                       |-- 파일시스템
                       |-- 코드 실행

=== 보안 가이드라인 ===
  1. 자격 증명은 외부 도구에서만 관리 (샌드박스 안에 시크릿 넣지 마세요)
  2. Human-in-the-Loop -- 민감한 작업에 사람 승인 요구
  3. 네트워크 접근 차단 -- 불필요한 아웃바운드 연결 차단
  4. 아웃바운드 모니터링 -- 예기치 않은 외부 연결 감시
  5. 출력 검토 -- 샌드박스 출력을 적용 전 검토


## [2-5] : ACP (Agent Communication Protocol)

**ACP**는 코딩 에이전트와 개발 환경(에디터/IDE) 간의 통신을 표준화하는 프로토콜입니다.

### MCP vs ACP 구분

| 프로토콜 | 용도 | 방향 |
|---------|------|------|
| **MCP** (Model Context Protocol) | 에이전트 <-> 외부 서비스(DB, API, 검색) | 에이전트가 **도구를 호출** |
| **ACP** (Agent Communication Protocol) | 에이전트 <-> 에디터/IDE | 에디터가 **에이전트를 호출** |

### ACP 서버 구현 예시

```python
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    checkpointer=MemorySaver(),
)

server = AgentServerACP(agent)
server.run()  # stdio 모드
```

### ACP 지원 에디터

| 에디터 | 통합 방식 |
|--------|----------|
| **Zed** | 네이티브 통합 |
| **JetBrains IDEs** | 빌트인 지원 |
| **VS Code** | vscode-acp 플러그인 |
| **Neovim** | ACP 호환 플러그인 |

In [16]:
# [7-16] : ACP 서버 구성 및 에디터 설정 예시 (참고용)
# [핵심] ACP는 에디터·외부 시스템과 Agent를 연결하는 통신 프로토콜
# [주의] 실행 전 API Key 설정을 확인하세요
print("=== ACP 서버 구현 예시 ===")
print()

acp_server_code = """\
# acp_server.py
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    checkpointer=MemorySaver(),
)

server = AgentServerACP(agent)
server.run()  # stdio 모드
"""
print(acp_server_code)

print("설치: pip install deepagents-acp")
print("실행: python acp_server.py")
print()

# Zed 에디터 설정 예시
print("=== Zed 에디터 설정 (settings.json) ===")
zed_config = '''\
{
  "agent_servers": [
    {
      "command": "python",
      "args": ["acp_server.py"],
      "env": { "OPENAI_API_KEY": "sk-..." }
    }
  ]
}
'''
print(zed_config)

=== ACP 서버 구현 예시 ===

# acp_server.py
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    checkpointer=MemorySaver(),
)

server = AgentServerACP(agent)
server.run()  # stdio 모드

설치: pip install deepagents-acp
실행: python acp_server.py

=== Zed 에디터 설정 (settings.json) ===
{
  "agent_servers": [
    {
      "command": "python",
      "args": ["acp_server.py"],
      "env": { "OPENAI_API_KEY": "sk-..." }
    }
  ]
}



## [2-6] : 실무 배포 구조 설계

하네스 + 백엔드 + 샌드박스 + ACP를 결합하면 **프로덕션 레디 에이전트**를 구성할 수 있습니다.

### 통합 아키텍처

```
┌──────────────┐     ACP      ┌───────────────────────────────┐
│  에디터/IDE   │ <----------> │         AgentHarness          │
│  (Zed, VSC)  │              │                               │
└──────────────┘              │  ┌─────────┐  ┌────────────┐  │
                              │  │Planning │  │  Memory    │  │
                              │  │write_   │  │ AGENTS.md  │  │
                              │  │  todos  │  │ SKILL.md   │  │
                              │  └─────────┘  └────────────┘  │
                              │                               │
                              │  ┌── CompositeBackend ───────┐│
                              │  │ /memories/ -> StoreBackend││
                              │  │ /workspace/-> Filesystem  ││
                              │  │ /scratch/ -> StateBackend ││
                              │  └──────────────────────────-┘│
                              │           |                    │
                              │      API 호출                  │
                              │           v                    │
                              │  ┌────────────────┐            │
                              │  │   Sandbox      │            │
                              │  │  (Modal/E2B)   │            │
                              │  │  코드 실행      │            │
                              │  └────────────────┘            │
                              └───────────────────────────────┘
```

In [17]:
# [7-17] : 프로덕션 에이전트 통합 구성 예시 (참고용)
# [핵심] SKILL.md는 Agent 행동을 선언적으로 정의하는 명세서
# [주의] 프로덕션에서는 샌드박스 내부에 시크릿을 전달하지 마세요
production_config = """\
from deepagents import create_deep_agent
from deepagents.backends import (
    StateBackend, FilesystemBackend, StoreBackend, CompositeBackend,
)
from deepagents.backends.sandbox import ModalSandbox
from deepagents_acp import AgentServerACP
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. CompositeBackend 팩토리
def production_backend(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),       # 영속 메모리
            "/workspace/": FilesystemBackend(          # 로컬 파일
                root_dir="./workspace", virtual_mode=True
            ),
        },
    )


# 2. 에이전트 생성 (하네스 조립)
agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 시니어 개발자입니다.",
    backend=production_backend,
    store=InMemoryStore(),
    checkpointer=MemorySaver(),
    memory=["/memory/AGENTS.md"],       # 항상 로드
    skills=["/skills/"],                # 온디맨드 스킬
    interrupt_on={"execute": True},      # 코드 실행 전 HITL
)


# 3. ACP 서버로 에디터 연결
server = AgentServerACP(agent)
server.run()
"""

print("=== 프로덕션 에이전트 통합 구성 ===")
print(production_config)

print("이 구성의 효과:")
print("  1. AGENTS.md -- 프로젝트 규칙 항상 적용")
print("  2. SKILL.md -- 필요 시 전문 지식 온디맨드 로드")
print("  3. CompositeBackend -- 메모리 영속 + 파일 접근 + 스크래치")
print("  4. HITL -- 코드 실행 전 사람 승인")
print("  5. ACP -- 에디터에서 직접 에이전트 제어")

=== 프로덕션 에이전트 통합 구성 ===
from deepagents import create_deep_agent
from deepagents.backends import (
    StateBackend, FilesystemBackend, StoreBackend, CompositeBackend,
)
from deepagents.backends.sandbox import ModalSandbox
from deepagents_acp import AgentServerACP
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. CompositeBackend 팩토리
def production_backend(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),       # 영속 메모리
            "/workspace/": FilesystemBackend(          # 로컬 파일
                root_dir="./workspace", virtual_mode=True
            ),
        },
    )


# 2. 에이전트 생성 (하네스 조립)
agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 시니어 개발자입니다.",
    backend=production_backend,
    store=InMemoryStore(),
    checkpointer=MemorySaver(),
    memory=["/memory/AGENTS.md"],       # 항상 로드
    skills=["/skills/"]

---
# [정리] : 핵심 요약

## Part 1 -- SKILL.md 설계 + 하네스 아키텍처

| 항목 | 핵심 내용 |
|------|----------|
| **AgentHarness** | Planning + Filesystem + Memory + Skills + Context 관리를 하나로 조립 |
| **write_todos** | 복잡한 작업을 구조화된 태스크 리스트로 분해 |
| **파일시스템 7종** | ls, read_file, write_file, edit_file, glob, grep, execute |
| **컨텍스트 오프로딩** | 20K 토큰 초과 시 디스크 저장 + 포인터 참조 |
| **AGENTS.md** | 항상 로드 (Always-on), 프로젝트 규칙/컨벤션 |
| **SKILL.md** | 온디맨드 로드, Progressive Disclosure 패턴 |

## Part 2 -- 스토리지 백엔드 + 샌드박스 + 배포

| 항목 | 핵심 내용 |
|------|----------|
| **BackendProtocol** | 모든 백엔드가 구현하는 플러그인 인터페이스 |
| **4가지 백엔드** | State(에페메럴) / Filesystem(디스크) / Store(크로스스레드) / Composite(라우팅) |
| **CompositeBackend** | /memories/ -> StoreBackend, /workspace/ -> FilesystemBackend |
| **Sandbox-as-Tool** | 에이전트 외부에서 샌드박스 API 호출 (권장 패턴) |
| **ACP** | 에디터-에이전트 통신 표준 (MCP와 보완 관계) |
| **배포 구조** | Harness + CompositeBackend + Sandbox + ACP = 프로덕션 에이전트 |

### 확장 방향
-> **08_MiniProject_해설.ipynb** : 하네스 엔지니어링을 활용한 미니 프로젝트